# Лабораторная работа 8: параметрические эксперименты

В этой части исследуется чувствительность DES-SIR к параметрам `beta`,
`contacts`, `gamma`, а также сравниваются экспоненциальная и фиксированная
длительности болезни.

In [1]:
using DrWatson
@quickactivate "project"

ENV["GKSwstype"] = "100"

using CSV
using DataFrames
using Plots

include(srcdir("SIRDESLab08.jl"))
using .SIRDESLab08

mkpath(datadir("sims"))
mkpath(plotsdir())

u0 = [990, 10, 0]
base_parameters = [0.05, 10.0, 0.25]
tmax = 40.0

40.0

## Чувствительность к параметрам

In [2]:
scan_beta = sensitivity_scan(
    u0,
    base_parameters;
    parameter = :beta,
    values = [0.03, 0.05, 0.07],
    tmax = tmax,
    seed = 2100,
)
scan_contacts = sensitivity_scan(
    u0,
    base_parameters;
    parameter = :contacts,
    values = [6.0, 10.0, 14.0],
    tmax = tmax,
    seed = 2200,
)
scan_gamma = sensitivity_scan(
    u0,
    base_parameters;
    parameter = :gamma,
    values = [0.15, 0.25, 0.40],
    tmax = tmax,
    seed = 2300,
)

sensitivity_summary = vcat(scan_beta.summary, scan_contacts.summary, scan_gamma.summary)
CSV.write(datadir("sims", "sir_sensitivity_summary.csv"), sensitivity_summary)

println("Анализ чувствительности DES-SIR")
println(sensitivity_summary)

function plot_infected_scan(scan, parameter_label, filename)
    chart = plot(
        xlabel = "Время",
        ylabel = "Инфицированные",
        title = "Чувствительность к параметру $parameter_label",
    )
    for trajectory in scan.trajectories
        plot!(
            chart,
            trajectory.data.t,
            trajectory.data.I;
            label = "$parameter_label = $(trajectory.value)",
            linewidth = 2,
        )
    end
    savefig(chart, plotsdir(filename))
end

plot_infected_scan(scan_beta, "beta", "sir_sensitivity_beta.png")
plot_infected_scan(scan_contacts, "c", "sir_sensitivity_contacts.png")
plot_infected_scan(scan_gamma, "gamma", "sir_sensitivity_gamma.png")

Анализ чувствительности DES-SIR
9×14 DataFrame
 Row │ scenario     parameter  value    beta     contacts  gamma    recovery_mode  peak_infected  peak_time  final_susceptible  final_infected  final_recovered  affected_share  events 
     │ String       String     Float64  Float64  Float64   Float64  String         Int64          Float64    Int64              Int64           Int64            Float64         Int64  
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ sensitivity  beta          0.03     0.03      10.0     0.25  exponential               20    2.41634                860               4              136           0.14      266
   2 │ sensitivity  beta          0.05     0.05      10.0     0.25  exponential              147   22.6253                 273              14              713           0.727    1430
   3 │ sensitivity  beta      

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/plots/sir_sensitivity_gamma.png"

## Фиксированная длительность болезни

В дополнительном эксперименте время до выздоровления заменяется на
детерминированную величину `1 / gamma`. Это убирает хвост
экспоненциального распределения и делает выздоровления более синхронными.

In [3]:
recovery_modes = compare_recovery_modes(u0, base_parameters; tmax = tmax, seed = 3100)
recovery_summary = vcat(recovery_modes.exponential.summary, recovery_modes.fixed.summary)
CSV.write(datadir("sims", "sir_recovery_modes_summary.csv"), recovery_summary)

println("Сравнение длительности болезни")
println(recovery_summary)

plot(
    recovery_modes.exponential.data.t,
    recovery_modes.exponential.data.I;
    label = "Экспоненциальная длительность",
    linewidth = 2,
    xlabel = "Время",
    ylabel = "Инфицированные",
    title = "Влияние длительности болезни",
)
plot!(
    recovery_modes.fixed.data.t,
    recovery_modes.fixed.data.I;
    label = "Фиксированная длительность",
    linewidth = 2,
    linestyle = :dash,
)
savefig(plotsdir("sir_recovery_modes.png"))

Сравнение длительности болезни
2×14 DataFrame
 Row │ scenario       parameter  value    beta     contacts  gamma    recovery_mode  peak_infected  peak_time  final_susceptible  final_infected  final_recovered  affected_share  events 
     │ String         String     Float64  Float64  Float64   Float64  String         Int64          Float64    Int64              Int64           Int64            Float64         Int64  
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ recovery_mode  base           NaN     0.05      10.0     0.25  exponential              204    23.7704                164              41              795           0.836    1621
   2 │ recovery_mode  base           NaN     0.05      10.0     0.25  fixed                    288    11.6792                228               0              772           0.772    1534


"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/plots/sir_recovery_modes.png"

## Сводные показатели

In [4]:
metric_plots = plot(layout = (1, 2), size = (1000, 420))
levels = 1:3
level_ticks = (levels, ["низкий", "базовый", "высокий"])
for (parameter_name, marker_shape) in zip(["beta", "contacts", "gamma"], [:circle, :diamond, :utriangle])
    rows = sensitivity_summary[sensitivity_summary.parameter .== parameter_name, :]
    plot!(
        metric_plots[1],
        levels,
        rows.peak_infected;
        marker = marker_shape,
        label = parameter_name,
        xticks = level_ticks,
        xlabel = "Уровень параметра",
        ylabel = "Пик I",
        title = "Максимум числа инфицированных",
    )
    plot!(
        metric_plots[2],
        levels,
        rows.affected_share;
        marker = marker_shape,
        label = parameter_name,
        xticks = level_ticks,
        xlabel = "Уровень параметра",
        ylabel = "Итоговая доля заболевших",
        title = "Размер эпидемии",
    )
end
savefig(metric_plots, plotsdir("sir_sensitivity_metrics.png"))

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/plots/sir_sensitivity_metrics.png"